In [0]:
%sql
create or replace table northwell.development.omny_fix_sample_medications_scm as
with spp as (
  SELECT 
    p.NDCCode,
    -- /* using fallback logic for non-numeric NDC codes, verify if this needed or a drop is preferred */ 
    -- CASE
    --   WHEN `NDCCode` NOT REGEXP '^[0-9]+$' THEN COALESCE(`DrugCatalogKey`, `NDCCode`)
    --   ELSE `NDCCode`
    -- END AS `FinalNDCCode`,
    p.ProductID,
    -- order by to get stable mappings
    ROW_NUMBER() OVER (PARTITION BY p.ProductID ORDER BY p.Active DESC, p.IsRepackaged ASC, p.CreatedWhen DESC, p.TouchedWhen DESC, p.ProductPackageID DESC, p.NDCCode DESC) AS rn 
  FROM _bronze.prod01_uat.dbo_sxammproductpackage p
), 
oto as (
  SELECT
    distinct
    p.NDCCode,
    otoa.RXCUI_CD,
    ord.GUID,
    row_number() over (partition by ord.GUID order by otoa.IsLVP asc, p.rn asc, p.ndccode asc) as rn
  FROM
    _bronze.prod01_uat.dbo_cv3ordertaskoccurrence task
  INNER JOIN _bronze.prod01_uat.dbo_cv3ordertask ot ON task.OrderTaskGUID = ot.GUID
  INNER JOIN _bronze.prod01_uat.dbo_cv3order ord ON ord.GUID = ot.OrderGUID
  INNER JOIN _bronze.prod01_uat.dbo_cv3medicationextension me ON ord.GUID = me.GUID
  INNER JOIN (
    SELECT
      otoa.OrderTaskOccurrenceGUID,
      otoa.ProductID,
      -- otoa.StockItemProductID,
      genItem.RxNormCode as RXCUI_CD,
      genItem.IsLVP,
      ROW_NUMBER() OVER (
        PARTITION BY otoa.OrderTaskOccurrenceGUID
        order by
          genItem.IsLVP asc, genItem.TouchedWhen desc, otoa.ProductID asc, genItem.RxNormCode ASC -- added
      ) as ranker
    FROM
      _bronze.prod01_uat.dbo_sxammordertaskoccurrenceadmin otoa
    
      JOIN _bronze.prod01_uat.dbo_sxammproduct pr on pr.ProductID = otoa.ProductID
    
      JOIN _bronze.prod01_uat.dbo_sxammgenericitem genItem ON pr.genericitemid = genItem.genericitemid
  ) otoa ON otoa.ranker = 1
  AND otoa.OrderTaskOccurrenceGUID = task.GUID
  LEFT JOIN spp p ON otoa.ProductID = p.ProductID AND p.rn = 1
  where task.TaskStatusCode = "Performed"
),
mdm as (
    SELECT
        mdm.*,
        ROW_NUMBER() OVER(
            partition by NDC
            order by
                etl_load_ts DESC
        ) seq
    FROM
        _bronze.hca_rdm_meta_uat.rdm_ontology_mdctn_ndc_rxcui_gnrc mdm
),
silver_medications as (

  /* Inpatient Medications */
  select
    distinct m.medicationid as order_medication_identifier,
    e.encounter_identifier,
    e.patient_identifier,
    e.patient_epi,
    o.ordercreateddtm as order_date_time,
    t_loc.orgloc_department as ordering_department,
    pr.npi as authorizing_provider_npi,
    'Inpatient' as order_mode,
    oto.NDCCode as order_ndc,
    -- New field
    coalesce(
      mext.SMMDispenseInfo, 
      concat(cv3o.Name, ' (', mext.OrderedAs, ')')
      ) as medication_brand_name,
    -- 
    -- cv3o.Name as medication_brand_name,
    md.genericname as generic_name,
    dd.dosage as dose,
    ud.uom as dose_unit,
    sfd.frequency as frequency,
    mdm.PRODUCT_STRENGTH_DESC as strength,
    mfd.medform as form,
    rd.medroute as route,
    o.significantdtm as start_date_time,
    o.stopdtm as end_date_time,
    osd.comment as status,
    -- task.TaskStatusCode as status2,
    '' as reason_for_stopping,
    o.modifier as prescription_instructions,
    m.refills as refills,
    m.quantity as quantity,
    null as total_days_supplied,
    'e-prescribed' as order_class
    -- :week_num as week_num
from
    -- northwell.omny_silver.encounters_scm e
    -- NOTE: change back for historical data
    northwell.development.omny_fix_sample_temp_encounter_scm e 
    inner join _bronze.sca_acutecare_uat.dbo_scaorder o on o.visitid = e.encounter_identifier
    left join _bronze.prod01_uat.dbo_cv3order cv3o on cv3o.GUID = o.OrderGUID -- added
    inner join _bronze.sca_acutecare_uat.dbo_scamedication m on o.visitid = m.visitid
    and o.orderid = m.orderid
    left join northwell.omny_silver._temp_orgloc_scm t_loc on t_loc.locationdimid = o.requestedlocationdimid 
    left join northwell.omny_silver._temp_provider_scm pr on pr.id = o.providerdimid
    inner join _bronze.sca_acutecare_uat.dbo_scamedicationdim md on m.medicationdimid = md.medicationdimid
    inner join _bronze.sca_acutecare_uat.dbo_scamedformdim mfd on m.medformdimid = mfd.medformdimid
    inner join _bronze.sca_acutecare_uat.dbo_scaprescriptiontypedim ptd on m.prescriptiontypedimid = ptd.prescriptiontypedimid
    inner join _bronze.sca_acutecare_uat.dbo_scadosagedim dd on m.lowdosagedimid = dd.dosagedimid
    inner join _bronze.sca_acutecare_uat.dbo_scauomdim ud on ud.uomdimid = m.dosageuomdimid
    inner join _bronze.sca_acutecare_uat.dbo_scamedroutedim rd on rd.medroutedimid = m.medroutedimid
    inner join _bronze.sca_acutecare_uat.dbo_scafrequencydim sfd on sfd.frequencydimid = m.frequencydimid
    left join _bronze.sca_acutecare_uat.dbo_scaorderstatusdim osd on osd.orderstatusdimid = o.orderstatusdimid
    left join _bronze.prod01_uat.dbo_cv3medicationextension mext ON o.orderGUID = mext.GUID
    left join oto on oto.GUID = mext.GUID and oto.rn = 1
    left join mdm on oto.NDCCode = mdm.NDC and mdm.seq = 1
    
    -- left join _bronze.prod01_uat.dbo_cv3ordertaskoccurrence task
    --   on task.OrderGUID = cv3o.GUID
where
    o.isactive = 1
    and lower(md.medname) not in ('(adm override)', '(floorstock)')
    and m.isactive = 1
    and osd.comment not in (
        'Cancelled',
        'Cancelled via patient Discharge',
        'Cancelled via patient Transfer',
        'Hold',
        'Pending verification by Attending',
        'Pending verification by Nurse',
        'Pending verification by nurse 2'
    )

UNION ALL

/* Prescriptions */
select
    distinct m.medicationid as order_medication_identifier,
    e.encounter_identifier,
    e.patient_identifier,
    e.patient_epi,
    COALESCE(m.medicationDtm, p.startdtm, p.srcupdtdtm) as order_date_time,
    t_loc.orgloc_department as ordering_department,
    pr.npi as authorizing_provider_npi,
    'Outpatient' as order_mode,
    ndc.NDC as order_ndc,
    p.DrugName as medication_brand_name,
    md.genericname as generic_name,
    dd.dosage as dose,
    ud.uom as dose_unit,
    sfd.frequency as frequency,
    -- '' as strength,
    mdm.PRODUCT_STRENGTH_DESC as strength,
    mfd.medform as form,
    rd.medroute as route,
    p.StartDtm as start_date_time,
    p.EndDtm as end_date_time,
    sd.comment as status,
    '' as reason_for_stopping,
    '' as prescription_instructions,
    m.refills as refills,
    m.quantity as quantity,
    p.DurationAmount as total_days_supplied,
    ptd.prescriptionType as order_class
    -- :week_num as week_num
from
    -- northwell.omny_silver.encounters_scm e
    -- NOTE: change back for historical data
    northwell.development.omny_fix_sample_temp_encounter_scm e 
    inner join _bronze.sca_acutecare_uat.dbo_scamedication m on e.encounter_identifier = m.visitid
    inner join _bronze.sca_acutecare_uat.dbo_scaprescription p on m.PrescriptionID = p.PrescriptionID
    left join northwell.omny_silver._temp_orgloc_scm t_loc on t_loc.locationdimid = p.locationdimid 
    left join northwell.omny_silver._temp_provider_scm pr on pr.id = p.providerdimid
    inner join _bronze.sca_acutecare_uat.dbo_scamedicationdim md on m.medicationdimid = md.medicationdimid
    inner join _bronze.sca_acutecare_uat.dbo_scamedformdim mfd on m.medformdimid = mfd.medformdimid
    inner join _bronze.sca_acutecare_uat.dbo_scaprescriptiontypedim ptd on m.prescriptiontypedimid = ptd.prescriptiontypedimid
    inner join _bronze.sca_acutecare_uat.dbo_scadosagedim dd on m.lowdosagedimid = dd.dosagedimid
    inner join _bronze.sca_acutecare_uat.dbo_scauomdim ud on ud.uomdimid = m.dosageuomdimid
    inner join _bronze.sca_acutecare_uat.dbo_scamedroutedim rd on rd.medroutedimid = m.medroutedimid
    inner join _bronze.sca_acutecare_uat.dbo_scafrequencydim sfd on sfd.frequencydimid = m.frequencydimid
    left join _bronze.sca_acutecare_uat.dbo_scastatusdim sd on sd.statusdimid = p.statusdimid
    
    -- left join oto on oto.GUID = mext.GUID and oto.rn = 1
    -- left join mdm on oto.NDCCode = mdm.NDC and mdm.seq = 1

    -- using dbo_scamedicationndcdim
    left join (
        select 
            ndc.*, 
            row_number() over (partition by ndc.MedicationDimID order by ndc.MedicationNDCDimID desc) as rn 
        from 
            _bronze.sca_acutecare_uat.dbo_scamedicationndcdim ndc
    ) ndc on ndc.MedicationDimID = m.MedicationDimID and ndc.rn = 1
    left join mdm on try_cast(ndc.NDC as STRING) = try_cast(mdm.NDC as STRING) and mdm.seq = 1
where
    lower(md.medname) not in ('(adm override)', '(floorstock)')
    and m.isactive = 1

)

select * from silver_medications